# Comparar runs

Este notebook só **lê** as pastas `runs/*/`, sem treinar nada. Para atualizar, rode tudo de novo.

Cada run deixa na sua pasta:
- `metricas.csv`: a curva de loss, gravada pelo `train.py`;
- `resultado_test.json`: o resumo da avaliação, gravado pelo `evaluate.py`;
- `erros_test.csv`: as linhas erradas.

## 1. Carregar as runs

In [ ]:
import os
import json
import glob
import pandas as pd
import matplotlib.pyplot as plt

RUNS = []              # vazio = todas; ou uma lista, por exemplo ['baseline', 'relu']
BASELINE = 'baseline'  # a run com que as outras são comparadas
SPLIT = 'test'

PARTES = ['comum', 'ruido', 'anos_dentro', 'anos_fora_209x', 'anos_fora_2100', 'formato']
TIPOS = ['valida', 'fronteira', 'dia_perto', 'dia_longe', 'mes_perto', 'mes_longe']

curvas, resultados = [], []
for pasta in sorted(glob.glob('runs/*/')):
    nome = os.path.basename(os.path.normpath(pasta))
    if RUNS and nome not in RUNS:
        continue
    if os.path.exists(os.path.join(pasta, 'metricas.csv')):
        c = pd.read_csv(os.path.join(pasta, 'metricas.csv'))
        c['run'] = nome
        curvas.append(c)
    arquivo = os.path.join(pasta, f'resultado_{SPLIT}.json')
    if os.path.exists(arquivo):
        with open(arquivo, encoding='utf-8') as f:
            resultados.append(json.load(f))

curvas = pd.concat(curvas, ignore_index=True)
print(f"{curvas['run'].nunique()} runs com curva, {len(resultados)} com resultado")

## 2. O que muda em cada run

Mostra só as colunas da config que **variam** entre as runs. O resto é igual em todas.

In [ ]:
configs = pd.DataFrame({r['run']: r['config'] for r in resultados}).T
configs['params'] = [r['params'] for r in resultados]
varia = [c for c in configs.columns if configs[c].astype(str).nunique() > 1]
configs[varia]

## 3. Curvas de loss

- **Esquerda:** a loss de validação de todas as runs, em escala log. Mostra quem cai mais rápido e se a curva ainda caía no fim.
- **Direita:** treino contra validação de uma run. Se as duas se afastam, é overfitting.

In [ ]:
RUN_DETALHE = BASELINE

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4.5))
for nome, c in curvas.groupby('run'):
    a.plot(c['iter'], c['val_loss'], label=nome)
a.set_yscale('log')
a.set_xlabel('passo')
a.set_ylabel('loss de validação (log)')
a.set_title('Loss de validação')
a.legend(fontsize=8)

c = curvas[curvas['run'] == RUN_DETALHE]
b.plot(c['iter'], c['train_loss'], label='treino')
b.plot(c['iter'], c['val_loss'], label='validação')
b.set_yscale('log')
b.set_xlabel('passo')
b.set_title(f'{RUN_DETALHE}: treino x validação')
b.legend()
plt.tight_layout()
plt.show()

## 4. Tabela final

Uma linha por run, ordenada pela loss de validação (é por ela que se escolhe a melhor config, e não pelo teste).
Os acertos estão em %. A fronteira e o mês impossível são medidos **dentro do `comum`**, para não misturar com o `formato`.

In [ ]:
linhas = []
for r in resultados:
    pt = r['parte_tipo']['comum']
    linha = {
        'run': r['run'],
        'params': r['params'],
        'passos': r['config']['max_iters'],
        'segundos': curvas.loc[curvas['run'] == r['run'], 'segundos'].max(),
        'best_val_loss': r['best_val_loss'],
    }
    for p in PARTES:
        linha[p] = 100 * r['por_parte'].get(p, float('nan'))
    linha['fronteira (comum)'] = 100 * pt.get('fronteira', float('nan'))
    linha['mês imp. (comum)'] = 100 * (pt.get('mes_perto', 0) + pt.get('mes_longe', 0)) / 2  # média simples dos dois
    linhas.append(linha)

tabela = pd.DataFrame(linhas).set_index('run').sort_values('best_val_loss')
METRICAS = PARTES + ['fronteira (comum)', 'mês imp. (comum)']
tabela.style.format({'params': '{:,.0f}', 'passos': '{:,.0f}', 'segundos': '{:.0f}',
                     'best_val_loss': '{:.4f}', **{m: '{:.1f}' for m in METRICAS}})

## 5. Acerto por parte do teste, lado a lado

In [ ]:
ax = tabela[PARTES].T.plot.bar(figsize=(12, 4.5), width=0.8)
ax.set_ylabel('acerto exato (%)')
ax.set_ylim(0, 100)
ax.set_title('Acerto por parte do teste')
ax.legend(fontsize=8, ncol=2)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Uma run por dentro: parte x tipo, e por campo

- **Mapa:** o acerto de cada tipo de data dentro de cada parte. É a leitura correta do "por tipo".
- **Tabela por campo:** onde está o erro: no ano, no mês, no dia ou só na marca `T`/`F`.

In [ ]:
RUN_MAPA = BASELINE

r = next(r for r in resultados if r['run'] == RUN_MAPA)
mapa = pd.DataFrame(r['parte_tipo']).reindex(index=TIPOS, columns=PARTES) * 100

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(mapa, cmap='RdYlGn', vmin=0, vmax=100)
ax.set_xticks(range(len(PARTES)), PARTES, rotation=30, ha='right')
ax.set_yticks(range(len(TIPOS)), TIPOS)
for i in range(mapa.shape[0]):
    for j in range(mapa.shape[1]):
        v = mapa.iat[i, j]
        ax.text(j, i, '' if pd.isna(v) else f'{v:.1f}', ha='center', va='center', fontsize=8)
fig.colorbar(im, label='acerto (%)')
ax.set_title(f'{RUN_MAPA}: acerto por parte e tipo')
plt.tight_layout()
plt.show()

por_campo = pd.DataFrame(r['por_campo']).reindex(columns=PARTES).T * 100
por_campo.style.format('{:.1f}')

## 7. Ablação: diferença para a baseline

Cada célula é a diferença, em pontos percentuais, entre a run e a baseline. Verde é melhor e vermelho é pior.
Diferenças pequenas podem ser só sorte: compare com o desvio entre seeds (seção 8).

In [ ]:
if BASELINE in tabela.index:
    delta = tabela[METRICAS] - tabela.loc[BASELINE, METRICAS]
    tabela_delta = delta.drop(BASELINE).style.format('{:+.1f}').background_gradient(cmap='RdYlGn', vmin=-10, vmax=10)
else:
    tabela_delta = f"a run '{BASELINE}' ainda não existe"
tabela_delta

## 8. Seeds: média e desvio

As runs com o mesmo nome e sufixos `_s1`, `_s2`, `_s3` são a mesma config com sementes diferentes.
O desvio mostra quanto o resultado varia só por sorte.

In [ ]:
familia = tabela.index.str.replace(r'_s\d+$', '', regex=True)
grupos = tabela.groupby(familia)
n_seeds = grupos.size()
media = grupos[METRICAS].mean()
desvio = grupos[METRICAS].std()
resumo = media.round(1).astype(str) + ' ± ' + desvio.round(1).astype(str)
resumo['n_seeds'] = n_seeds
resumo[resumo['n_seeds'] > 1]

## 9. Exemplos de erro

In [ ]:
RUN_ERROS = BASELINE
PARTE_ERROS = 'comum'   # comum, ruido, formato, anos_dentro ou anos_fora
N = 10

erros = pd.read_csv(f'runs/{RUN_ERROS}/erros_{SPLIT}.csv', dtype=str, keep_default_na=False)
sel = erros[erros['parte'] == PARTE_ERROS]
print(f"{len(sel):,} erros na parte {PARTE_ERROS}")
sel.sample(min(N, len(sel)), random_state=0)[['input', 'previsto', 'target', 'tipo', 'formato', 'ruido', 'ano']]